In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Suppress TensorFlow warnings
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

print("TensorFlow version:", tf.__version__)

# --- STEP 1: LOAD DATA ---
FILE_NAME = 'energy_data_3_years.csv'
try:
    data = pd.read_csv(FILE_NAME)
except FileNotFoundError:
    print(f"Error: The file '{FILE_NAME}' was not found.")
    exit()

# Prepare data
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values('Date').reset_index(drop=True)

print("Data loaded successfully!")
print(f"Date range: {data['Date'].min().date()} to {data['Date'].max().date()}")
print(f"Total records: {len(data)}")

# --- STEP 2: PREPARE SEQUENCES ---
def create_sequences(data, seq_length=60):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

SEQ_LENGTH = 60  # Use 60 days to predict next day

# --- STEP 3: PROCESS SOLAR DATA ---
print(f"\n{'='*70}")
print("SOLAR ENERGY - LSTM FORECASTING")
print(f"{'='*70}")

solar_col = 'Solar_Energy_Produced_kWh'
solar_data = data[solar_col].fillna(data[solar_col].mean()).values.reshape(-1, 1)

# Normalize
solar_scaler = MinMaxScaler(feature_range=(0, 1))
solar_scaled = solar_scaler.fit_transform(solar_data)

print(f"Mean: {data[solar_col].mean():.2f} kWh")
print(f"Std: {data[solar_col].std():.2f} kWh")

# Create sequences
X_solar, y_solar = create_sequences(solar_scaled, SEQ_LENGTH)
print(f"\nSequences created: {len(X_solar)} training samples")

# --- STEP 4: BUILD AND TRAIN SOLAR LSTM MODEL ---
print("\nBuilding LSTM model...")
solar_model = Sequential([
    LSTM(50, activation='relu', return_sequences=True, input_shape=(SEQ_LENGTH, 1)),
    Dropout(0.2),
    LSTM(50, activation='relu', return_sequences=False),
    Dropout(0.2),
    Dense(25, activation='relu'),
    Dense(1)
])

solar_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
print("✓ Model compiled")

print("\nTraining LSTM (Solar)...")
history = solar_model.fit(
    X_solar, y_solar,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)
print(f"✓ Training complete!")
print(f"Final Loss: {history.history['loss'][-1]:.6f}")

# --- STEP 5: FORECAST 2026 SOLAR ---
print("\nGenerating 2026 forecast...")

# Use last 60 days as starting point
last_sequence = solar_scaled[-SEQ_LENGTH:].reshape(1, SEQ_LENGTH, 1)
solar_forecast_2026 = []

for day in range(365):
    pred = solar_model.predict(last_sequence, verbose=0)[0, 0]
    solar_forecast_2026.append(pred)
    
    # Update sequence
    new_seq = np.append(last_sequence[0, 1:, 0], pred)
    last_sequence = new_seq[-SEQ_LENGTH:].reshape(1, SEQ_LENGTH, 1)

# Denormalize
solar_forecast_2026 = np.array(solar_forecast_2026).reshape(-1, 1)
solar_forecast_2026_denorm = solar_scaler.inverse_transform(solar_forecast_2026)
solar_forecast_2026_denorm = np.maximum(solar_forecast_2026_denorm, 0)

print(f"✓ Forecast generated!")
print(f"\nSolar Forecast Statistics (2026):")
print(f"Mean: {solar_forecast_2026_denorm.mean():.2f} kWh")
print(f"Std: {solar_forecast_2026_denorm.std():.2f} kWh")
print(f"Min: {solar_forecast_2026_denorm.min():.2f} kWh")
print(f"Max: {solar_forecast_2026_denorm.max():.2f} kWh")

# Create output dataframe
dates_2026 = pd.date_range('2026-01-01', '2026-12-31', freq='D')
forecast_df = pd.DataFrame({
    'Date': dates_2026,
    'Solar_Energy_Forecast_kWh': solar_forecast_2026_denorm.flatten()
})

# --- STEP 6: WIND ENERGY FORECAST ---
wind_col = 'Wind_Energy_Produced_kWh'
if wind_col in data.columns:
    print(f"\n{'='*70}")
    print("WIND ENERGY - LSTM FORECASTING")
    print(f"{'='*70}")
    
    wind_data = data[wind_col].fillna(data[wind_col].mean()).values.reshape(-1, 1)
    
    print(f"Mean: {data[wind_col].mean():.2f} kWh")
    print(f"Std: {data[wind_col].std():.2f} kWh")
    
    # Normalize
    wind_scaler = MinMaxScaler(feature_range=(0, 1))
    wind_scaled = wind_scaler.fit_transform(wind_data)
    
    # Create sequences
    X_wind, y_wind = create_sequences(wind_scaled, SEQ_LENGTH)
    print(f"Sequences created: {len(X_wind)} samples")
    
    # Build wind model
    print("\nBuilding LSTM model for wind...")
    wind_model = Sequential([
        LSTM(50, activation='relu', return_sequences=True, input_shape=(SEQ_LENGTH, 1)),
        Dropout(0.2),
        LSTM(50, activation='relu', return_sequences=False),
        Dropout(0.2),
        Dense(25, activation='relu'),
        Dense(1)
    ])
    
    wind_model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    print("\nTraining LSTM (Wind)...")
    wind_history = wind_model.fit(
        X_wind, y_wind,
        epochs=20,
        batch_size=32,
        validation_split=0.1,
        verbose=0
    )
    print(f"✓ Training complete!")
    
    # Forecast wind
    print("\nGenerating wind forecast...")
    last_wind_seq = wind_scaled[-SEQ_LENGTH:].reshape(1, SEQ_LENGTH, 1)
    wind_forecast_2026 = []
    
    for day in range(365):
        pred = wind_model.predict(last_wind_seq, verbose=0)[0, 0]
        wind_forecast_2026.append(pred)
        
        new_seq = np.append(last_wind_seq[0, 1:, 0], pred)
        last_wind_seq = new_seq[-SEQ_LENGTH:].reshape(1, SEQ_LENGTH, 1)
    
    # Denormalize
    wind_forecast_2026 = np.array(wind_forecast_2026).reshape(-1, 1)
    wind_forecast_2026_denorm = wind_scaler.inverse_transform(wind_forecast_2026)
    wind_forecast_2026_denorm = np.maximum(wind_forecast_2026_denorm, 0)
    
    # Add to forecast
    forecast_df['Wind_Energy_Forecast_kWh'] = wind_forecast_2026_denorm.flatten()
    
    print(f"✓ Wind forecast completed!")
    print(f"Mean: {wind_forecast_2026_denorm.mean():.2f} kWh")
    print(f"Max: {wind_forecast_2026_denorm.max():.2f} kWh")

# --- STEP 7: SAVE AND DISPLAY RESULTS ---
print(f"\n{'='*70}")
print("FORECAST RESULTS")
print(f"{'='*70}")

# Save forecast
output_file = 'lstm_forecast_2026.csv'
forecast_df.to_csv(output_file, index=False)
print(f"✓ Forecast saved to '{output_file}'")

# Display sample
print(f"\nForecast Sample (First 10 days):")
print(forecast_df.head(10).to_string(index=False))

# Monthly summary
forecast_df['Month'] = forecast_df['Date'].dt.to_period('M')
monthly = forecast_df.groupby('Month')['Solar_Energy_Forecast_kWh'].agg(['mean', 'min', 'max', 'sum'])

print(f"\n{'='*70}")
print("MONTHLY SUMMARY (2026)")
print(f"{'='*70}")
print(monthly.to_string())

# Total energy
total_solar = forecast_df['Solar_Energy_Forecast_kWh'].sum()
print(f"\nTotal Solar Energy (2026): {total_solar:,.2f} kWh")

if 'Wind_Energy_Forecast_kWh' in forecast_df.columns:
    total_wind = forecast_df['Wind_Energy_Forecast_kWh'].sum()
    total_combined = total_solar + total_wind
    print(f"Total Wind Energy (2026): {total_wind:,.2f} kWh")
    print(f"Combined Total (2026): {total_combined:,.2f} kWh")
    print(f"\nSolar: {(total_solar/total_combined)*100:.1f}%")
    print(f"Wind: {(total_wind/total_combined)*100:.1f}%")

print(f"\n{'='*70}")
print("✓ LSTM FORECASTING COMPLETE")
print(f"{'='*70}")

TensorFlow version: 2.19.0
Data loaded successfully!
Date range: 2022-12-10 to 2025-12-08
Total records: 1095

SOLAR ENERGY - LSTM FORECASTING
Mean: 9.53 kWh
Std: 2.01 kWh

Sequences created: 1035 training samples

Building LSTM model...
✓ Model compiled

Training LSTM (Solar)...
✓ Training complete!
Final Loss: 0.010774

Generating 2026 forecast...
✓ Training complete!
Final Loss: 0.010774

Generating 2026 forecast...
✓ Forecast generated!

Solar Forecast Statistics (2026):
Mean: 6.81 kWh
Std: 0.01 kWh
Min: 6.81 kWh
Max: 6.89 kWh

WIND ENERGY - LSTM FORECASTING
Mean: 3.93 kWh
Std: 3.69 kWh
Sequences created: 1035 samples

Building LSTM model for wind...

Training LSTM (Wind)...
✓ Forecast generated!

Solar Forecast Statistics (2026):
Mean: 6.81 kWh
Std: 0.01 kWh
Min: 6.81 kWh
Max: 6.89 kWh

WIND ENERGY - LSTM FORECASTING
Mean: 3.93 kWh
Std: 3.69 kWh
Sequences created: 1035 samples

Building LSTM model for wind...

Training LSTM (Wind)...
✓ Training complete!

Generating wind forecast.